In [0]:
%sql
USE CATALOG edgar;
CREATE SCHEMA IF NOT EXISTS stg;
CREATE SCHEMA IF NOT EXISTS fct;
CREATE SCHEMA IF NOT EXISTS mrt;
USE SCHEMA stg;
CREATE TABLE IF NOT EXISTS stg_submissions_raw;
CREATE TABLE IF NOT EXISTS edgar.stg.ingestion_control (
    cik STRING,
    payload_hash STRING,
    max_acceptance_datetime STRING,
    last_fetch_at TIMESTAMP
);
CREATE VOLUME IF NOT EXISTS edgar.stg.vol;

## Submissions Data

In [0]:
import hashlib
import json
import requests
import io
from datetime import datetime, timezone
import pyspark.sql.functions as F

In [0]:
headers = {"User-Agent": "User Agent"}
nvidia = requests.get("https://data.sec.gov/submissions/CIK0001045810.json", headers=headers)
data = nvidia.json()

# Stable serialization + content hash for idempotency
payload_str = json.dumps(data, sort_keys=True)
payload_hash = hashlib.sha256(payload_str.encode()).hexdigest()
max_acc = max(data["filings"]["recent"]["acceptanceDateTime"])

# Look up the last ingested hash for this CIK
cik = str(data["cik"])
ctrl_rows = spark.table("edgar.stg.ingestion_control").filter(F.col("cik") == cik).collect()
stored_hash = ctrl_rows[0]["payload_hash"] if ctrl_rows else None
changed = stored_hash != payload_hash
print(f"Hash {'differs from' if changed else 'matches'} stored hash — payload {'will be' if changed else 'will NOT be'} ingested.")


In [0]:
if changed:
    # (i) write raw JSON file to the volume
    dbutils.fs.put(f"/Volumes/edgar/stg/vol/raw_submissions/CIK{cik}.json", payload_str, overwrite=True)
    print("File uploaded successfully!")

    # (ii) append snapshot to stg_submissions_raw
    stg_df = spark.createDataFrame([{
        "cik": cik,
        "ticker": ",".join(data.get("tickers", [])),
        "company_name": data["name"],
        "raw_payload": payload_str,
        "ingested_at": datetime.now(timezone.utc)
    }])
    stg_df.write.mode("append").saveAsTable("edgar.stg.stg_submissions_raw", mergeSchema=True)

    # (iii) upsert the control row
    spark.sql(f"""
        MERGE INTO edgar.stg.ingestion_control AS t
        USING (SELECT '{cik}' AS cik, '{payload_hash}' AS payload_hash,
                      '{max_acc}' AS max_acceptance_datetime,
                      current_timestamp() AS last_fetch_at) AS s
        ON t.cik = s.cik
        WHEN MATCHED THEN UPDATE SET t.payload_hash = s.payload_hash,
                                    t.max_acceptance_datetime = s.max_acceptance_datetime,
                                    t.last_fetch_at = s.last_fetch_at
        WHEN NOT MATCHED THEN INSERT (cik, payload_hash, max_acceptance_datetime, last_fetch_at)
        VALUES (s.cik, s.payload_hash, s.max_acceptance_datetime, s.last_fetch_at)
    """)
    print("New payload ingested: file written, stg row appended, control updated.")
else:
    print(f"No change since last fetch for CIK {cik} — skipping file write, stg append, and control update.")